# mjo_wavenum_freq_season

- "Calculates wavenumber-frequency spectra via seasonal averaging as defined by the US-CLIVAR MJO diagnostics website"
- [NCL Reference](https://www.ncl.ucar.edu/Document/Functions/Diagnostics/mjo_wavenum_freq_season.shtml)

### Example NCL Script and Output

- mjo_wavenum_freq_season.ncl
- mjo_output/mjo_wavenum_freq_season_output.txt

In [1]:
import os
import numpy as np
import xarray as xr
from datetime import datetime

In [2]:
wavenum_freq = np.loadtxt("mjo_output/mjo_wavenum_freq_season_winter_ncl_output.txt", skiprows=18)
ncl_wavenum_freq = wavenum_freq.reshape(289, 181)
print(f"NCL Wavenumbers: {len(ncl_wavenum_freq)}")
print(f"NCL Frequencies: {len(ncl_wavenum_freq[0])}")

NCL Wavenumbers: 289
NCL Frequencies: 181


MJO CLIVAR: Wave number-frequency spectra
- "winter": 180 days (starts November 1)
- "summer": 180 days (starts May 1)
- "summer": 365 days (starts January 1)

In [3]:
u850_data = xr.open_dataset(os.getcwd() + "/data/anomaly/QBOi.EXP1.AMIP.001.u850.day.anom.nc")
u850_data

<xarray.Dataset> Size: 565MB
Dimensions:  (time: 2555, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 20kB ...
    U850     (time, lat, lon) float32 565MB ...

In [4]:
# Filter out latitude and time ranges (based on NCL script)

## filter out latitude ranges
latS = -10
latN = 10

u850_data = u850_data.sel(lat=slice(latS, latN)) 

## filter out time ranges 
twStrt = "1979-01-01" # time window start
twLast = "1981-12-31" # time window end

u850_data = u850_data.sel(time=slice(twStrt, twLast)) 
u850_data

<xarray.Dataset> Size: 28MB
Dimensions:  (time: 1095, lat: 22, lon: 288)
Coordinates:
  * time     (time) object 9kB 1979-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 9kB ...
    U850     (time, lat, lon) float32 28MB ...

In [5]:
# Average data over latitude and use the averaged to compute spectra
## Compute the average of latitude

def dim_avg_n_wrap_python(data, dim):
    # https://www.ncl.ucar.edu/Document/Functions/Contributed/dim_avg_n_Wrap.shtml
    data = data.mean(dim=dim)
    return data

u850_data = dim_avg_n_wrap_python(u850_data, "lat")
u850_data

<xarray.Dataset> Size: 1MB
Dimensions:  (time: 1095, lon: 288)
Coordinates:
  * time     (time) object 9kB 1979-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 9kB 1e+04 1e+04 1e+04 1e+04 ... 3e+04 3e+04 3e+04
    U850     (time, lon) float32 1MB -1.226 -1.366 -1.502 ... -0.8523 -0.9687

MJO wavenumber-frequency spectra based on Level 2 diagnostics- Based on [US-CLIVAR MJO Working Group (2009) "MJO Simulation Diagnostics - Level 2 diagnostics"](https://doi.org/10.1175/2008JCLI2731.1)

> "Level 2 diagnostics are designed to explore more detailed features of the MJO. They include wavenumber-frequency spectra of individual fields, cross-spectral quantities between different fields, and a multivariate EOF analysis. Wavenumber-frequency spectra for equatorial precipitation and 850-hPa zonal wind are shown in Fig. 7 for boreal summer, and in Fig. 8 for boreal winter.

> The spectra were computed by Fourier transforming 180-day segments centered on boreal summer and boreal winter, forming power, and then averaging over all years of data (1979–2005). The resulting bandwidth is (180 days)−1. Only the climatological season cycle was removed before calculation of the spectra.

> By definition, eastward propagation is represented by positive frequency and positive wavenumber whereas westward propagation is represented with one or the other of the frequency or wavenumber being negative. If standing oscillations are present, they will project as equal amounts of power in eastward and westward directions. The results indicate a concentration of power at 30–90-day periods and zonal wavenumber 1 for 850-hPa zonal wind, and zonal wavenumbers 1–3 for precipitation and OLR (e.g., Salby and Hendon 1994)"

In [6]:
seasonName = "winter"

In [7]:
def mjo_wavenum_freq_season_as_python(input_data, data_var, seasonName):
    # Python equivalent of mjo_wavenum_freq_season()
    # Based on ncl: mjo_wavenum_freq_season
    # https://github.com/NCAR/ncl/blob/8f9e9476281cc6f6d9d12eaa78729c7003ca24b7/ni/src/examples/gsun/diagnostics_cam.ncl#L2886
    data = input_data[data_var]

    seasonName = seasonName.lower()
    print(f"Date for {seasonName}")
    if seasonName == "winter":
        # filter out boreal winter across multiple years
        # Winter: November, December, January, February, March, April
        data = data.sel(time=data.time.dt.month.isin([11, 12, 1, 2, 3, 4]))
    if seasonName == "summer":
        # filter out boreal summer across multiple years
        # Summer: May, June, July, August, September, October
        data = data.sel(time=data.time.dt.month.isin([5, 6, 7, 8, 9, 10]))

    time_step_dt = (data.time[1] - data.time[0]).astype('timedelta64[D]').item().days # time steps (in days)
    lon_step_dx = (data.lon[1] - data.lon[0]).item() # longitude (degree step)
    print(f"Time step: {time_step_dt} day(s) with lon step: {lon_step_dx} degrees")
        
    # Detrend data (over time)
    poly_coeffs = data.polyfit(dim="time", deg=lon_step_dx, skipna=True)
    fit = xr.polyval(data["time"], poly_coeffs["polyfit_coefficients"])
    data_detrend = data - fit
    data_detrend.attrs = data.attrs
    data_detrend.name = data.name

    # remove seasonal cycle from data before apply 2D FFT
    ## calculate mean seasonal climate for each season across time range
    seasonal_climatology_mean = data_detrend.groupby("time.season").mean("time")
    ## remove seasonal cycle (deseasonalize) from data
    data_deseason = data_detrend.groupby("time.season") - seasonal_climatology_mean

    n_time = len(data_deseason.time)
    m_lon = len(data_deseason.lon)

    # average Wavenumber-Frequency power spectra over seasons via 2D FFT
    ## 2D FFT on input data for specific season
    spectrum = np.fft.fft2(data_deseason.values)
    #spectrum = np.fft.fftn(data_deseason)
    power_spectral_density = np.abs(spectrum)**2 / (n_time * m_lon) ## Power Spectrum is the square magntidue
    
    ## Determine frequency
    freqs = np.fft.fftfreq(n_time, d=time_step_dt)
    
    ## Determine wavenumbers
    wavenumbers = np.fft.fftfreq(m_lon, d=lon_step_dx)
    
    ## Shift the zero frequency to the center
    wavenumbers = np.fft.fftshift(wavenumbers)
    freqs = np.fft.fftshift(freqs)
    power_spectral_density = np.fft.fftshift(power_spectral_density)
    
    ## Return data as Wavenumber X Frequency
    wavenum_freq = xr.DataArray(power_spectral_density,
                                 coords={"frequency": freqs, 
                                         "wavenumbers": wavenumbers},
                                 dims=["frequency", "wavenumbers"],
                                 name="wavenum_freq")

    print(f"[wavenumber | {len(wavenum_freq.wavenumbers)}] x [freq | {len(wavenum_freq.frequency)}]")
    return wavenum_freq

In [8]:
wavenum_freq = mjo_wavenum_freq_season_as_python(u850_data, "U850", seasonName)
wavenum_freq

Date for winter
Time step: 1 day(s) with lon step: 1.25 degrees
[wavenumber | 288] x [freq | 543]


<xarray.DataArray 'wavenum_freq' (frequency: 543, wavenumbers: 288)> Size: 1MB
array([[2.12695480e-04, 2.12127893e-05, 4.27927990e-06, ...,
        6.59781007e-06, 1.68671068e-04, 2.55947221e-05],
       [3.84491550e-05, 5.80130912e-06, 8.19025089e-06, ...,
        8.82042974e-05, 7.15346731e-05, 2.28862332e-05],
       [4.51168354e-05, 1.45202624e-05, 7.39055391e-05, ...,
        1.29501890e-04, 7.41282742e-06, 5.09076405e-05],
       ...,
       [4.51168354e-05, 5.09076405e-05, 7.41282742e-06, ...,
        2.52699861e-05, 7.39055391e-05, 1.45202624e-05],
       [3.84491550e-05, 2.28862332e-05, 7.15346731e-05, ...,
        1.18400149e-05, 8.19025089e-06, 5.80130912e-06],
       [2.12695480e-04, 2.55947221e-05, 1.68671068e-04, ...,
        6.24579169e-05, 4.27927990e-06, 2.12127893e-05]], shape=(543, 288))
Coordinates:
  * frequency    (frequency) float64 4kB -0.4991 -0.4972 ... 0.4972 0.4991
  * wavenumbers  (wavenumbers) float64 2kB -0.4 -0.3972 ... 0.3944 0.3972

In [9]:
# save output
ds = wavenum_freq.to_dataframe()
ds.to_csv(f"mjo_output/mjo_wavenum_freq_season_{seasonName}_python_output.txt", index=True)

In [80]:
def _taper(ts, alpha=0.1, iopt=0):
    # https://github.com/princekx/SEAPy_Climate/blob/ff48807c245978e85737cec778165c5faa7fc90b/src/diags_level2.py#L78
    if all(x == ts[0] for x in ts):
        print("all values are equal")
        iopt = 1
    if iopt == 0:
        tsmean = np.mean(ts)
    else:
        tsmean = 0.
    n = len(ts)
    m = max(1, int(alpha * n + 0.5) / 2)
    pim = np.pi / m

    tst = ts.copy()
    for i in range(1, int(m) + 1):
        weight = 0.5 - 0.5 * np.cos(pim * (i - 0.5))
        tst[i - 1] = (ts[i - 1] - tsmean) * weight + tsmean
        tst[n - i] = (ts[n - i] - tsmean) * weight + tsmean
    return tst

def resolveWavesHayashi(varfft, nDayWin, spd=1):
    # https://github.com/princekx/SEAPy_Climate/blob/ff48807c245978e85737cec778165c5faa7fc90b/src/diags_level2.py#L181
    N, mlon = varfft.shape
    pee = np.ones([N + 1, mlon + 1]) * -999
    varfft = np.abs(varfft) ** 2
    n_i = int(N / 2)
    mlon_i = int(mlon / 2)
    pee[ : n_i, : mlon_i] = varfft[n_i : N, mlon_i : 0:-1]
    pee[n_i : , : mlon_i ] = varfft[ : n_i + 1, mlon_i : 0 : -1]
    pee[ : n_i + 1, mlon_i : ] = varfft[ n_i : : -1, : mlon_i + 1]
    pee[n_i + 1 : , mlon_i : ] = varfft[N - 1: n_i - 1: -1 , : mlon_i + 1]
    return pee

In [92]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # supress FutureWarning

def mjo_wavenum_freq_season_pv2(input_data, season_name):
    # Based on SEAPy_climate: https://github.com/princekx/SEAPy_Climate/blob/ff48807c245978e85737cec778165c5faa7fc90b/src/diags_level2.py#L755
    # And NCL: https://github.com/NCAR/ncl/blob/8f9e9476281cc6f6d9d12eaa78729c7003ca24b7/ni/src/examples/gsun/diagnostics_cam.ncl#L2886
   
    season_name = season_name.lower()
    ntime = len(input_data.time)
    mlon = len(input_data.lon)
    
    time = input_data.time
    time = input_data.indexes["time"].to_datetimeindex(unsafe=True) # convert to datetime object (generates futurewarning)
    # convert YYYY-MM-DD to YYYYMMDD
    time = time.strftime("%Y%m%d")

    # Winter Start -- November 1 [180 days]
    # Summer Start -- May 1 [180 days]
    # Annual Start -- Jan 1 [180 days]
    if season_name == "winter":
        # November 1
        mmStart = 11
        ddStart = 1 
        nDay = 180
    if season_name == "summer":
        # May 1
        mmStart = 5
        ddStart = 1
        nDay = 180
    if season_name == "annual":
        # January 1
        mmStart = 1
        ddStart = 1
        nDay = 365

    print(f"{season_name}: starts on {mmStart}/{ddStart} for {nDay} days")

    # Collect start dates
    is_start_date_mask = (input_data["time.month"] == mmStart) & (input_data["time.day"] == ddStart)
    iSea = input_data.where(is_start_date_mask, drop=True) # places where it is start date
    #iSea = [date_object.strftime("%Y%m%d") for date_object in iSea.time.values]
    iSea = iSea.time.values
    nYear = is_start_date_mask.sum(dim="time").item() # number of seasons
    print(f"Number of times start date {mmStart}/{ddStart} appears out of {len(input_data.time)} = {nYear}")

    #;*****************************************************************
    #; For a specific season, calculate spectra via averaging 
    #; over each seasonal segment.
    #; MJO Clivar says "no" to detrending/tapering.
    #; Hence, the following are just 'place holders'
    #;*****************************************************************
    
    # detrend overall series in time dimension
    # dtrend_leftdim(x, False)
    lon_step_dx = (input_data.lon[1] - input_data.lon[0]).item() # longitude (degree step)
    poly_coeffs = input_data.polyfit(dim="time", deg=lon_step_dx, skipna=True)
    fit = xr.polyval(input_data["time"], poly_coeffs["date_polyfit_coefficients"])
    data_detrend = input_data - fit # remove linear trend

    # Initalize
    power = np.zeros([mlon + 1, nDay + 1])
    work = data_detrend["U850"].values
    xAvgSea = 0.0
    xVarSea = 0.0 # variance (raw)
    xVarTap = 0.0 # variance after tapering
    kSea = 0 # count seasons used

    for ny in range(nYear):
        print(f"Season Date: {iSea[ny]}")
        index_obj = input_data.time.to_index()

        # get index of starting and ending point
        iStrt = index_obj.get_loc(iSea[ny]) # start index for current season
        iLast = iStrt + nDay # last index for current season

        # only run FFT if end index is within time index
        if iLast < ntime - 1:
            xSeason = work[iStrt:iLast,:]
            xAvg = np.average(xSeason) # season average over all time and longitude
            xSeason -= xAvg # remove season mean from time-lon season
            xVarSea += np.var(xSeason) # overall variance (raw)
            kSea += 1 # increment count of the seasons

            # tapering
            for lon in range(mlon):
                taper = _taper(xSeason[:, lon])
                xSeason[:, lon] = taper

            # variance after tapering
            xVarTap += np.var(xSeason)

            # FFT
            FFT_2d = xSeason.copy()
            FFT_2d = np.fft.fft2(xSeason.T) / mlon / nDay

            # Shift FFT
            power += resolveWavesHayashi(FFT_2d, nDay, spd=1)  # (wave,freq)

    # pooled seasonal variance
    xVarSea /= kSea
    xVarTap /= kSea

    power = np.ma.masked_array(power / kSea) # pooled spectra
    wavenum = np.arange(-mlon / 2, mlon / 2 + 1, 1)
    freq = np.linspace(-1 * nDay / 2, nDay / 2, nDay + 1) / nDay

    return power, wavenum, freq

In [95]:
power, wavenum, freq = mjo_wavenum_freq_season_pv2(u850_data, "winter")
print(power)
print(wavenum)
print(freq)

winter: starts on 11/1 for 180 days
Number of times start date 11/1 appears out of 1095 = 3
Season Date: 1979-11-01 00:00:00
Season Date: 1980-11-01 00:00:00
Season Date: 1981-11-01 00:00:00
[[1.99706793e-09 7.40990726e-10 8.20611550e-10 ... 8.20611550e-10
  7.40990726e-10 1.99706793e-09]
 [6.31012042e-11 1.32317578e-09 8.98681978e-10 ... 7.73069853e-10
  1.30461378e-10 6.31012042e-11]
 [2.18977041e-09 7.32338998e-10 7.06843676e-10 ... 1.03392682e-09
  1.63354059e-09 2.18977041e-09]
 ...
 [2.18977041e-09 1.63354059e-09 1.03392682e-09 ... 7.06843676e-10
  7.32338998e-10 2.18977041e-09]
 [6.31012042e-11 1.30461378e-10 7.73069853e-10 ... 8.98681978e-10
  1.32317578e-09 6.31012042e-11]
 [1.99706793e-09 7.40990726e-10 8.20611550e-10 ... 8.20611550e-10
  7.40990726e-10 1.99706793e-09]]
[-144. -143. -142. -141. -140. -139. -138. -137. -136. -135. -134. -133.
 -132. -131. -130. -129. -128. -127. -126. -125. -124. -123. -122. -121.
 -120. -119. -118. -117. -116. -115. -114. -113. -112. -111. -1

# mjo_wavenum_freq_season_plot

- "Plot wavenumber-frequency spectra as returned by mjo_wavenum_freq_season"
- [NCL Reference](https://www.ncl.ucar.edu/Document/Functions/Diagnostics/mjo_wavenum_freq_season_plot.shtml)

### Example Plot fom `mjo_wavenum_freq_season.ncl`

- mjo_output/mjo_wavenum_freq_season_plot.winter.png

<center>
<img src="mjo_output/mjo_wavenum_freq_season_plot.winter.png" width="400" height="600">
</center>

In [10]:
wavenum_freq = np.loadtxt(f"mjo_output/mjo_wavenum_freq_season_{seasonName}_ncl_output.txt", skiprows=18)
wavenum_freq

array([1.997126e-09, 7.409889e-10, 8.206045e-10, ..., 8.206045e-10,
       7.409889e-10, 1.997126e-09], shape=(52309,))

In [11]:
# Load NCL data to generate a comparison plot
wavenum_freq = np.loadtxt(f"mjo_output/mjo_wavenum_freq_season_{seasonName}_ncl_output.txt", skiprows=18)
ncl_wavenum_freq = wavenum_freq.reshape(289, 181)
ncl_wavenum_freq_xr = xr.DataArray(ncl_wavenum_freq,
                                   coords={"frequency": 
                                   dims=["frequency", "wavenumbers"],
                                   name="wavenum_freq")
ncl_wavenum_freq_xr.sel(frequency=slice(-0.5, 0.5)) 
#ncl_wavenum_freq_xr

SyntaxError: closing parenthesis ')' does not match opening parenthesis '{' on line 5 (1300787121.py, line 7)

In [ ]:
import matplotlib.pyplot as plt

def mjo_wavenum_freq_season_plot_as_python(plt_title, wavenum_freq,
                                           max_wavenum=6,
                                           min_freq=-0.5, max_freq=0.5):
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # filter wave numbers from 0 to 6 (default)
    wavenum_freq = wavenum_freq.sel(wavenumbers=slice(0, max_wavenum)) 
    # filter frequencies from -0.5 to 0.5 (default)
    wavenum_freq = wavenum_freq.sel(frequency=slice(min_freq, max_freq)) 

    # Plot
    plt.contour(wavenum_freq,
                vmax=(wavenum_freq).max(), vmin=(wavenum_freq).min(),
                levels=10)
    plt.imshow(wavenum_freq, 
               vmax=(wavenum_freq).max(), vmin=(wavenum_freq).min(), 
               aspect="auto")
    
    #ax.set_xlim([x_min, x_max])
    #ax.set_ylim([y_min, y_max])
    plt.title(f"Wavenumber-Frequency: {plt_title}")
    plt.xlabel("Frequency (cycles/day)")
    plt.ylabel("Zonal Wavenumber (cycles/degree)")
    #plt.grid(True)
    plt.show()

In [ ]:
mjo_wavenum_freq_season_plot_as_python(f"{seasonName.upper()}", ncl_wavenum_freq_xr)

In [ ]:
#mjo_wavenum_freq_season_plot_as_python(f"NCL -> {seasonName.upper()}", ncl_wavenum_freq)

In [ ]:
## Old Version:
import matplotlib.pyplot as plt

def mjo_wavenum_freq_season_plot_as_python(seasonName, power_spectrum):
    fig, ax = plt.subplots(figsize=(6, 6))
    power_spectrum.plot.pcolormesh()
    plt.title(f"Wavenumber-Frequency: {seasonName.upper()}")
    plt.xlabel("Frequency (cycles/day)")
    plt.ylabel("Zonal Wavenumber (cycles/degree)")
    #plt.grid(True)
    plt.show()